In [ ]:
import os
import pandas as pd
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

train_csv_path = "/kaggle/input/kyrgyz-anpr-kyrgyz-license-plate-recognition-challenge/train.csv"
df = pd.read_csv(train_csv_path)
df.head()

/kaggle/input/competitions/kyrgyz-anpr-kyrgyz-license-plate-recognition-challenge/train.csv
/kaggle/input/competitions/kyrgyz-anpr-kyrgyz-license-plate-recognition-challenge/test/KGS1840Q27613.png
/kaggle/input/competitions/kyrgyz-anpr-kyrgyz-license-plate-recognition-challenge/test/KGO6246AT19720.png
/kaggle/input/competitions/kyrgyz-anpr-kyrgyz-license-plate-recognition-challenge/test/01KG764ABJ19382.png
/kaggle/input/competitions/kyrgyz-anpr-kyrgyz-license-plate-recognition-challenge/test/KG2534SA25092.png
/kaggle/input/competitions/kyrgyz-anpr-kyrgyz-license-plate-recognition-challenge/test/KGN1616AA22554.png
/kaggle/input/competitions/kyrgyz-anpr-kyrgyz-license-plate-recognition-challenge/test/KG432BB22975.png
/kaggle/input/competitions/kyrgyz-anpr-kyrgyz-license-plate-recognition-challenge/test/KGB1111AI115.png
/kaggle/input/competitions/kyrgyz-anpr-kyrgyz-license-plate-recognition-challenge/test/KG4380BC28695.png
/kaggle/input/competitions/kyrgyz-anpr-kyrgyz-license-plate-recogn

In [3]:
import os
print(os.listdir("/kaggle/input/competitions"))

['kyrgyz-anpr-kyrgyz-license-plate-recognition-challenge']


In [4]:
import pandas as pd

# 完整单行路径，不要换行
folder_name = "kyrgyz-anpr-kyrgyz-license-plate-recognition-challenge"
ROOT = "/kaggle/input/competitions/" + folder_name + "/"

# 读取csv
train_df = pd.read_csv(ROOT + "train.csv")
print("数据形状：", train_df.shape)
print(train_df.head())

all_chars = set()
for label in train_df["label"]:
    all_chars.update(list(label))
char_list = sorted(list(all_chars))

char2idx = {c:i+1 for i, c in enumerate(char_list)}
idx2char = {i+1:c for i, c in enumerate(char_list)}
BLANK = 0
VOCAB_SIZE = len(char2idx) + 1

print("词汇表大小：", VOCAB_SIZE)
print("所有字符：", char_list)

数据形状： (35071, 2)
                        path         label
0   train/01KG203ABS7291.png  01<KG>203ABS
1    train/KGB3470C25709.png    <KG>B3470C
2    train/KG2774SK15923.png    <KG>2774SK
3   train/KGS7930AS21566.png   <KG>S7930AS
4  train/01KG161AAC14642.png  01<KG>161AAC
词汇表大小： 39
所有字符： ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '<', '>', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


In [5]:
import cv2
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# 图像预处理
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

class PlateDataset(Dataset):
    def __init__(self, df, img_root, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.img_root + row["path"]
        label_str = row["label"]

        # 读取图片并resize到(256,32)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (256, 32))
        img = Image.fromarray(img)

        if self.transform is not None:
            img = self.transform(img)
            
            
        # 文字转数字编码
        label_code = [char2idx[c] for c in label_str]
        return img, torch.LongTensor(label_code)

ModuleNotFoundError: No module named 'torch'

In [ ]:
temp_ds = PlateDataset(train_df, ROOT, train_transform)
img, lab = temp_ds[0]
print("图像类型:", type(img))

In [ ]:
print(train_transform)

In [ ]:
import torch
def collate_fn(batch):
    imgs, labels_list = zip(*batch)
    imgs= torch.stack(imgs, dim=0)

    label_flat = []
    label_len = []
    for lab in labels_list:
        if torch.is_tensor(lab):
            lab = lab.tolist()
        label_flat += lab
        label_len.append(len(lab))
    labels = torch.LongTensor(label_flat)
    label_lengths = torch.LongTensor(label_len)
    input_lengths = torch.full((imgs.size(0),), 32, dtype=torch.long)
    return imgs, labels, label_lengths, input_lengths

print("测试collate_fn")
test_batch = [temp_ds[i] for i in range(4)]
imgs, labels, label_lens, input_lens = collate_fn(test_batch)
print(f"图像批次形状:{imgs.shape}")
print(f"标签展平长度:{len(labels)}")
print(f"每个样本标签长度:{label_lens.tolist()}")
print(f"输入序列长度:{input_lens.tolist()}")

In [ ]:
from sklearn.model_selection import train_test_split

train_split, val_split = train_test_split(train_df, test_size=0.1, random_state=42)

train_dataset = PlateDataset(train_split, ROOT, train_transform)
val_dataset = PlateDataset(val_split, ROOT, train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

print("训练loader构建完成")

In [ ]:
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)
print("验证loader构建完成")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # CNN 特征提取
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1),
            nn.ReLU(),
            nn.MaxPool2d((2,2)),

            nn.Conv2d(64, 128, 3, 1, 1),
            nn.ReLU(),
            nn.MaxPool2d((2,2)),

            nn.Conv2d(128, 256, 3, 1, 1),
            nn.ReLU(),
            nn.Conv2d(256, 256, 3, 1, 1),
            nn.ReLU(),

            nn.Conv2d(256, 512, 3, 1, 1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.Conv2d(512, 512, 3, 1, 1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d((2,1)),
        )
        # RNN序列建模
        self.rnn = nn.GRU(
            input_size = 512*4,
            hidden_size = 256,
            num_layers =2, 
            bidirectional=True, 
            batch_first=False,
            dropout = 0.2
        )
        # 输出分类
        self.fc = nn.Linear(512, vocab_size)

    def forward(self, x):
        B, C, H, W = x.shape
        x = self.cnn(x)
        x = x.permute(3, 0, 1, 2).contiguous()
        x = x.view(x.size(0),x.size(1), -1)
        x, _ = self.rnn(x)
        x = self.fc(x)
        x = F.log_softmax(x, dim = -1)
        return x

# 初始化模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CRNN(VOCAB_SIZE).to(device)

# CTC损失 + 优化器
criterion = nn.CTCLoss(blank=0,zero_infinity=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print(f"CRNN模型初始化完毕")
print(f"使用设备:{device}")
print(f"模型参数量:{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print("\n 测试前向传播...")
with torch.no_grad():
    dummy_input = torch.randn(2, 3, 32, 256).to(device)
    dummy_output = model(dummy_input)
    print(f" 输入形状: {dummy_input.shape}")
    print(f" 输出形状: {dummy_output.shape}")
    print(f" 输出值范围: [{dummy_output.min():.2f}, {dummy_output.max():.2f}]")

In [ ]:
import torch
import torch.nn.functional as F

def ctc_decode(logits, idx2char):
    # 如果输入是2D，添加batch维度
    if len(logits.shape) == 2:
        logits = logits.unsqueeze(1)
    
    # 取最大概率的索引
    logits = F.log_softmax(logits, dim=-1)
    pred_idx = torch.argmax(logits, dim=-1)  # [T, B]
    pred_idx = pred_idx.transpose(0, 1).cpu()  # [B, T]
    
    results = []
    for seq in pred_idx:
        chars = []
        prev = -1
        for num in seq:
            # CTC解码：去重 + 跳过blank(0)
            if num != prev and num != 0:
                chars.append(idx2char[num])
            prev = num
        results.append(''.join(chars))
    
    return results

print("CTC解码函数已定义")

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    
    for imgs, labels, label_lengths,input_lengths in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        label_lengths = label_lengths.to(device)
        input_lengths = input_lengths.to(device)

        outputs = model(imgs)
        loss = criterion(outputs, labels, input_lengths, label_lengths)

        optimizer.zero_grad()
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 5.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)
print("train_epoch已定义")

def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    with torch.no_grad():
        for imgs, labels, label_lengths, input_lengths in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            input_lengths = input_length.to(device)

            outputs = model(imgs)
            loss = criterion(outputs, labels, input_lengths, label_lengths)
            total_loss += loss.item()

            preds = ctc_decode(outputs, idx2char)
            all_preds.extend(preds)
    avg_loss = total_loss / len(loader)
    return avg_loss, all_preds
print("val_epoch 已定义")

In [ ]:
EPOCHS = 25
best_val_loss = float('inf')

print("=" * 60)
print("开始训练")
print("=" * 60)

for epoch in range(EPOCHS):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"{'='*50}")
    
    
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, pred_list = val_epoch(model, val_loader, criterion, device)
    
    print(f"📊 Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    if pred_list:
        print(f"🔤 预测样例: {pred_list[:5]}")
    
    # ✅ 修复：在循环内部保存最佳模型
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_crnn.pth')
        print(f"✅ 保存最佳模型 (val_loss: {val_loss:.4f})")

print("\n" + "=" * 60)
print(f"🎉 训练完成！最佳验证损失: {best_val_loss:.4f}")
print("=" * 60)